
# HODGE v10a.30 — M5-FIRST Blind \(m_5/m_6/m_7\) A100 Runner

This notebook **does not execute the v10a.26 notebook** and does **not rerun the completed fourth-order/full-\(T_1\) calculation**.

It uses the v10a.26 source only as a definitions library. A dependency-closed bootstrap is extracted with Python's AST, while all top-level fourth-order operator moments, finite-cluster production, rooted subtraction, and `shape_cache` execution are excluded.

The first scientific calculation is the order-5 test:

1. certify the order-5 Haar sectors;
2. run order-5 SW/BCH and odd-depth regressions;
3. perform an order-5 one-face physical smoke test;
4. enumerate order-5 histories directly — **no order-4 support census**;
5. run resumable blind \(m_5\) rooted-cluster production;
6. freeze the result before external comparison.

The published \(m_5,m_6,m_7\) targets are absent.


In [ ]:

from __future__ import annotations

import ast
import builtins
import hashlib
import json
import os
import platform
import shutil
import subprocess
import symtable
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import nbformat
import numpy as np
import opt_einsum as oe
import sympy as sp

# ------------------------------ USER SETTINGS ------------------------------
USE_GOOGLE_DRIVE = True
WORKDIR_NAME = "HODGE_BLIND_M5_M7_M5_FIRST"
AUTO_UPLOAD_MISSING_FILES = True

V26_NAME = "NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb"
V28_NAME = "ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py"

# A100 production: zero means no per-invocation limit / no between-shape stop.
M5_MAX_NEW_SHAPES = 0
M5_TIME_BUDGET_MINUTES = 0
GPU_SW_MIN_DIM = 64
HERM_AUDIT_PAIRS = 24
DUPLICATE_CHECKS = 1
HEARTBEAT_SECONDS = 20

print("UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())


In [ ]:

# Safe GPU check: never replace CuPy inside a live runtime.
# Installing an arbitrary CuPy wheel can create a CUDA runtime/driver mismatch.
try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    )
    print("nvidia-smi:", smi.stdout.strip())
except Exception as exc:
    raise RuntimeError("No usable NVIDIA runtime detected. Select the A100 runtime first.") from exc

try:
    import cupy as cp
    gpu_count = int(cp.cuda.runtime.getDeviceCount())
    if gpu_count < 1:
        raise RuntimeError("CuPy reports zero CUDA devices")
    props = cp.cuda.runtime.getDeviceProperties(0)
    GPU_NAME = props["name"].decode() if isinstance(props["name"], (bytes, bytearray)) else str(props["name"])
    free_b, total_b = cp.cuda.runtime.memGetInfo()
except Exception as exc:
    raise RuntimeError(
        "The NVIDIA device is present but the installed CuPy build cannot use it. "
        "Restart with the working A100 image; this notebook intentionally does not pip-install/replace CuPy."
    ) from exc

print("CuPy:", cp.__version__)
print("GPU:", GPU_NAME)
print(f"GPU memory: {free_b / 2**30:.2f} / {total_b / 2**30:.2f} GiB free/total")


In [ ]:

# Durable checkpoints.
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        WORKDIR = Path("/content/drive/MyDrive") / WORKDIR_NAME
    except Exception as exc:
        print("Drive mount unavailable; using /content:", repr(exc))
        WORKDIR = Path("/content") / WORKDIR_NAME
else:
    WORKDIR = Path("/content") / WORKDIR_NAME

WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR = WORKDIR / "sources"
BOOTSTRAP_DIR = WORKDIR / "bootstrap"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)
print("WORKDIR:", WORKDIR)


In [ ]:

# Locate or upload the two source artifacts. They are copied into Drive for reproducibility.
CONTENT = Path("/content")

def locate(name: str) -> Path | None:
    for p in (CONTENT / name, SOURCE_DIR / name, WORKDIR / name, Path.cwd() / name):
        if p.exists():
            return p
    return None

def upload_if_missing(name: str) -> Path:
    p = locate(name)
    if p is not None:
        return p
    if not AUTO_UPLOAD_MISSING_FILES:
        raise FileNotFoundError(name)
    try:
        from google.colab import files
    except Exception as exc:
        raise FileNotFoundError(f"{name} missing and Colab upload is unavailable") from exc
    print(f"Upload {name}")
    uploaded = files.upload()
    if name in uploaded:
        return CONTENT / name
    if len(uploaded) == 1:
        src = CONTENT / next(iter(uploaded))
        dst = CONTENT / name
        if src != dst:
            shutil.move(str(src), str(dst))
        return dst
    raise RuntimeError(f"Expected {name}; received {list(uploaded)}")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

V26_PATH = upload_if_missing(V26_NAME)
V28_PATH = upload_if_missing(V28_NAME)

# Freeze exact source copies. Do not overwrite a different file with the same name silently.
for src in (V26_PATH, V28_PATH):
    dst = SOURCE_DIR / src.name
    if not dst.exists():
        shutil.copy2(src, dst)
    elif sha256_file(dst) != sha256_file(src):
        dst = SOURCE_DIR / f"{src.stem}_{sha256_file(src)[:12]}{src.suffix}"
        if not dst.exists():
            shutil.copy2(src, dst)

V26_SHA256 = sha256_file(V26_PATH)
V28_SHA256 = sha256_file(V28_PATH)
print("v10a.26:", V26_PATH, V26_SHA256)
print("v10a.28:", V28_PATH, V28_SHA256)



## Definitions-only v10a.26 bootstrap

The cell below parses v10a.26 and executes only the dependency closure needed by the order-generic engine. It does **not** call `IPython.run_cell`, does not execute the notebook top-to-bottom, and refuses any selected top-level statement carrying the known fourth-order production markers.


In [ ]:

# v10a.28 globals after removing the obsolete m4 one-face comparator dependency.
V26_BOOTSTRAP_ROOTS = (
    "L", "N", "faces", "verts", "T1_POLS", "anchor_faces", "V23C_ROOT",
    "V23C_POL", "_FAST_EPS", "oe", "LXState", "_V17_VAC",
    "_v17_apply_W_faces", "_v17_apply_W_labeled", "_v17_connected",
    "_v17_phys_index", "_v17_translate_support", "_v17_translate_face",
    "_v23c_split_h0", "_v23c_rooted_connected_subsets", "_v24c_shape_key",
    "_v24c_candidate_supports", "_v10a3_face_state",
    "_v10a3_physical_blocks", "_v10a3_compress_state", "_v9_flux_key_state",
    "_joint_canon_states", "lx_combine_bra_ket",
    "_v23_sw_exact", "_v23_sp", "_v23_random", "_V23CF",
    "_v26_singlet_multiplicity", "_V17_NEIGH",
)

# No extra legacy runtime seeds. The dependency closure pulls only symbols
# actually required by the order-generic engine.
V26_BOOTSTRAP_SEEDS = ()

_MUTATING_METHODS = {
    "add", "append", "clear", "discard", "extend", "insert", "pop",
    "remove", "setdefault", "sort", "update",
}


def _base_name(node):
    while isinstance(node, (ast.Attribute, ast.Subscript)):
        node = node.value
    return node.id if isinstance(node, ast.Name) else None


def _defined_or_mutated_names(stmt):
    names = set()

    class Visitor(ast.NodeVisitor):
        def visit_FunctionDef(self, node):
            names.add(node.name)
            # Definition-time expressions can depend on globals, but stores inside
            # the function body do not define module globals.
        visit_AsyncFunctionDef = visit_FunctionDef

        def visit_ClassDef(self, node):
            names.add(node.name)

        def visit_Import(self, node):
            for a in node.names:
                names.add(a.asname or a.name.split(".")[0])

        def visit_ImportFrom(self, node):
            for a in node.names:
                if a.name != "*":
                    names.add(a.asname or a.name)

        def visit_Name(self, node):
            if isinstance(node.ctx, (ast.Store, ast.Del)):
                names.add(node.id)

        def visit_Call(self, node):
            if isinstance(node.func, ast.Attribute) and node.func.attr in _MUTATING_METHODS:
                base = _base_name(node.func.value)
                if base:
                    names.add(base)
            self.generic_visit(node)

    Visitor().visit(stmt)
    return names


def _statement_dependencies(stmt, all_defined_names):
    """Global names referenced by one module-level statement, including functions."""
    text = ast.unparse(stmt)
    table = symtable.symtable(text, "<v26-bootstrap-statement>", "exec")
    deps = set()

    def walk(tab):
        module = tab.get_type() == "module"
        for sym in tab.get_symbols():
            if not sym.is_referenced():
                continue
            if module or sym.is_global():
                deps.add(sym.get_name())
        for child in tab.get_children():
            walk(child)

    walk(table)
    return deps & all_defined_names


def _is_import(stmt):
    return isinstance(stmt, (ast.Import, ast.ImportFrom))


def _is_definition(stmt):
    return isinstance(stmt, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))


def build_v26_bootstrap(notebook_path: Path, output_path: Path, manifest_path: Path):
    doc = nbformat.read(notebook_path, as_version=4)
    source = "\n\n".join(
        cell.source for cell in doc.cells
        if cell.cell_type == "code" and cell.source.strip()
    )
    tree = ast.parse(source, filename=str(notebook_path))
    statements = list(tree.body)

    defines = []
    defmap = {}
    for i, stmt in enumerate(statements):
        dn = _defined_or_mutated_names(stmt)
        defines.append(dn)
        for name in dn:
            defmap.setdefault(name, set()).add(i)

    all_names = set(defmap)
    selected = {i for i, stmt in enumerate(statements) if _is_import(stmt)}
    pending = list(dict.fromkeys((*V26_BOOTSTRAP_ROOTS, *V26_BOOTSTRAP_SEEDS)))
    resolved_names = set()

    while pending:
        name = pending.pop()
        if name in resolved_names:
            continue
        resolved_names.add(name)
        for idx in sorted(defmap.get(name, ())):
            if idx not in selected:
                selected.add(idx)
            for dep in _statement_dependencies(statements[idx], all_names):
                if dep not in resolved_names:
                    pending.append(dep)

    missing_defs = [name for name in V26_BOOTSTRAP_ROOTS if name not in defmap]
    if missing_defs:
        raise RuntimeError("v10a.26 source does not define required bootstrap names: " + ", ".join(missing_defs))

    # Fail closed if dependency extraction tries to pull a known production action.
    heavy_markers = (
        "FULL-T1 OPERATOR MOMENTS",
        "N=<R1|R1>",
        "J=<R1|R^2B>",
        "C1=<R1|RWRB>",
        "D=<WRB|RWRB>",
        "v10a.26 preflight: one-face Q2",
        "for ci, C in enumerate(CLUST",
        "for ci,C in enumerate(CLUST",
        "_v23c_fit_cluster(preC",
        "ROOTED INCIDENCE TRANSFORM",
        "shape_cache = _v26_load_checkpoint",
        "shape_cache=_v26_load_checkpoint",
    )
    rejected = []
    for idx in sorted(selected):
        stmt = statements[idx]
        if _is_definition(stmt) or _is_import(stmt):
            continue
        segment = ast.get_source_segment(source, stmt) or ast.unparse(stmt)
        hit = next((m for m in heavy_markers if m in segment), None)
        if hit:
            rejected.append((idx, getattr(stmt, "lineno", None), hit))
    if rejected:
        raise RuntimeError(f"Definitions-only bootstrap selected forbidden production statements: {rejected}")

    body = [statements[i] for i in sorted(selected)]
    module = ast.fix_missing_locations(ast.Module(body=body, type_ignores=[]))
    bootstrap_source = ast.unparse(module) + "\n"
    output_path.write_text(bootstrap_source, encoding="utf-8")

    manifest = {
        "schema": "hodge-v10a30-v26-definitions-only-bootstrap-v2",
        "source_notebook": str(notebook_path),
        "source_sha256": sha256_file(notebook_path),
        "bootstrap_sha256": hashlib.sha256(bootstrap_source.encode()).hexdigest(),
        "source_statement_count": len(statements),
        "selected_statement_count": len(body),
        "roots": list(V26_BOOTSTRAP_ROOTS),
        "seeds": list(V26_BOOTSTRAP_SEEDS),
        "selected": [
            {
                "index": i,
                "line": getattr(statements[i], "lineno", None),
                "type": type(statements[i]).__name__,
                "defines_or_mutates": sorted(defines[i]),
            }
            for i in sorted(selected)
        ],
        "forbidden_production_statements_selected": False,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


BOOTSTRAP_PATH = BOOTSTRAP_DIR / f"v26_defs_{V26_SHA256[:16]}.py"
BOOTSTRAP_MANIFEST = BOOTSTRAP_DIR / f"v26_defs_{V26_SHA256[:16]}.json"

_BOOTSTRAP_SCHEMA = "hodge-v10a30-v26-definitions-only-bootstrap-v2"
_rebuild_bootstrap = not BOOTSTRAP_PATH.exists() or not BOOTSTRAP_MANIFEST.exists()
if not _rebuild_bootstrap:
    try:
        info = json.loads(BOOTSTRAP_MANIFEST.read_text(encoding="utf-8"))
        _rebuild_bootstrap = any((
            info.get("schema") != _BOOTSTRAP_SCHEMA,
            info.get("source_sha256") != V26_SHA256,
            info.get("roots") != list(V26_BOOTSTRAP_ROOTS),
            info.get("seeds") != list(V26_BOOTSTRAP_SEEDS),
            sha256_file(BOOTSTRAP_PATH) != info.get("bootstrap_sha256"),
        ))
    except Exception:
        _rebuild_bootstrap = True
if _rebuild_bootstrap:
    info = build_v26_bootstrap(V26_PATH, BOOTSTRAP_PATH, BOOTSTRAP_MANIFEST)

print("Bootstrap selected statements:", info["selected_statement_count"], "/", info["source_statement_count"])
print("Bootstrap:", BOOTSTRAP_PATH)
print("Bootstrap SHA-256:", sha256_file(BOOTSTRAP_PATH))

# Force bootstrap configuration before executing its safe initializers.
os.environ["PREFER_GPU"] = "1"
os.environ["GLUE_L"] = "5"
os.environ["V10A23_CLUSTER_POL"] = "2"
os.environ["V10A23_CLUSTER_PROGRESS"] = "0"
os.environ["V10A7_SUPPORT_POLS"] = "2"
os.environ["V10A7_RECHECK_Q1"] = "0"
os.environ["V10A7_UNBLIND"] = "0"

bootstrap_source = BOOTSTRAP_PATH.read_text(encoding="utf-8")
exec(compile(bootstrap_source, str(BOOTSTRAP_PATH), "exec"), globals())

missing = [name for name in V26_BOOTSTRAP_ROOTS if name not in globals()]
if missing:
    raise RuntimeError("Definitions-only bootstrap missed required symbols: " + ", ".join(missing))

# These are signatures of the old production execution. They must not exist.
for forbidden_name in ("Dop", "K4op", "V26_RESULT", "M4_ORACLE", "shape_cache"):
    if forbidden_name in globals():
        raise RuntimeError(f"Forbidden m4 production state leaked into bootstrap: {forbidden_name}")

print("M4 production execution: SKIPPED")
print("v10a.26 definitions-only namespace: PASS")
print("L =", L, "N =", N, "faces =", len(faces))



## Patch v10a.28 to be order-native

The original v10a.28 source contains three compatibility checks inherited from development: an all-orders synthetic loop, a v10a.26 one-face \(O(u^4)\) comparator, and an order-4 support-census regression. This cell removes those checks and replaces them with current-order-only tests. The calculation and production kernels are otherwise unchanged.


In [ ]:

def patch_v28_m5_first(source: str) -> str:
    original = source

    required_markers = (
        'V28_SCHEMA = "hodge-v10a28-order-aware-krylov-gram-haar9-v1"',
        '    "_v24c_candidate_supports", "_v10a3_face_state", "_v10a3_h0_state_inner",\n',
        '    "_joint_canon_states", "lx_combine_bra_ket", "_v26_sw_blocks",\n',
        '    "_v26_singlet_multiplicity", "_V17_NEIGH", "_v23c_fit_cluster",\n',
        '    for order in (4, 5, 6, 7):\n',
        'print("watchdog                     : NONE")\n',
        '    cache = _v28_load()\n    representatives = {}\n',
    )
    missing_markers = [m for m in required_markers if m not in source]
    if missing_markers:
        raise RuntimeError(f"Unexpected v10a.28 source; patch anchors missing: {missing_markers}")

    # New checkpoint schema: never mix this M5-first run with old runner checkpoints.
    source = source.replace(
        'V28_SCHEMA = "hodge-v10a28-order-aware-krylov-gram-haar9-v1"',
        'V28_SCHEMA = "hodge-v10a30-m5-first-krylov-gram-haar9-v1"',
        1,
    )

    # The old comparator function is no longer a required global.
    source = source.replace(
        '    "_v26_singlet_multiplicity", "_V17_NEIGH", "_v23c_fit_cluster",\n',
        '    "_v26_singlet_multiplicity", "_V17_NEIGH",\n',
        1,
    )
    source = source.replace(
        '    "_v24c_candidate_supports", "_v10a3_face_state", "_v10a3_h0_state_inner",\n',
        '    "_v24c_candidate_supports", "_v10a3_face_state",\n',
        1,
    )
    source = source.replace(
        '    "_joint_canon_states", "lx_combine_bra_ket", "_v26_sw_blocks",\n',
        '    "_joint_canon_states", "lx_combine_bra_ket",\n',
        1,
    )

    # Synthetic band theorem regression only at the requested order.
    source = source.replace(
        '    for order in (4, 5, 6, 7):\n',
        '    for order in (V28_ORDER,):\n',
        1,
    )

    start_marker = 'print("\\n[2] ORDER-GENERIC SW REGRESSION")'
    end_marker = '# ---------------------------------------------------------------------------\n# 4. Generic bidirectional support history census'
    start = source.index(start_marker)
    end = source.index(end_marker, start)

    current_order_block = r'''
print(f"\n[2] ORDER-{V28_ORDER} SW/BCH REGRESSION")
V28_SW_REGRESSION_ERROR = _v28_sw_regression(V28_ORDER)
v28_gate(
    f"NumPy SW recursion matches exact rational BCH through O(u^{V28_ORDER})",
    V28_SW_REGRESSION_ERROR < V28_SW_TOL,
    f"max error={V28_SW_REGRESSION_ERROR:.3e}",
)
V28_GPU_SW_REGRESSION_ERROR = _v28_gpu_sw_regression(V28_ORDER)
v28_gate(
    f"order-{V28_ORDER} GPU and CPU SW/BCH backends agree",
    V28_GPU_SW_REGRESSION_ERROR < 2e-9,
    (f"max error={V28_GPU_SW_REGRESSION_ERROR:.3e}" if V28_GPU_ENABLED else "GPU unavailable; CPU fallback"),
)
V28_ORDER_BAND_ERRORS, V28_ODD_DEEP_SENSITIVITY = _v28_order_band_regression()
v28_gate(
    f"order-{V28_ORDER} Krylov truncation matches the full band model",
    V28_ORDER_BAND_ERRORS[V28_ORDER] < V28_SW_TOL,
    f"O{V28_ORDER}={V28_ORDER_BAND_ERRORS[V28_ORDER]:.2e}",
)
if V28_ORDER % 2:
    v28_gate(
        f"order-{V28_ORDER} regression detects removal of the deepest self-block",
        V28_ODD_DEEP_SENSITIVITY[V28_ORDER] > 1e-8,
        f"response={V28_ODD_DEEP_SENSITIVITY[V28_ORDER]:.2e}",
    )
else:
    v28_gate(
        f"order-{V28_ORDER} uses the even-order deepest-self omission theorem",
        V28_ORDER_BAND_ERRORS[V28_ORDER] < V28_SW_TOL,
        "no odd-order Q_d W Q_d block is required",
    )

print(f"\n[2b] ORDER-{V28_ORDER} ONE-FACE PHYSICAL SMOKE — NO M4 COMPARATOR")
_v28_one_face = frozenset((V23C_ROOT,))
V28_ONE_FACE = _v28_cluster_coefficients(_v28_one_face, V28_ORDER)
_v28_one_face_finite = bool(np.all(np.isfinite(np.asarray(V28_ONE_FACE["coef"]))))
_v28_one_face_ok = (
    _v28_one_face_finite
    and V28_ONE_FACE["one_herm"] < V28_HERM_TOL
    and V28_ONE_FACE["vac_herm"] < V28_HERM_TOL
    and V28_ONE_FACE["sw_offdiag"] < 2e-8
    and len(V28_ONE_FACE["coef"]) >= V28_ORDER + 1
)
v28_gate(
    f"order-{V28_ORDER} one-face physical Gram/SW smoke passes",
    _v28_one_face_ok,
    f"layers={V28_ONE_FACE['one_layers']}/{V28_ONE_FACE['vac_layers']}; "
    f"SWoff={V28_ONE_FACE['sw_offdiag']:.3e}; finite={_v28_one_face_finite}",
)
V28_ONE_FACE_PREFIX_ERROR = None

'''
    source = source[:start] + current_order_block + source[end:]

    # Reuse the order-native one-face smoke as the one-face production shape.
    # This avoids computing the same M5 cluster twice; no M4 data is involved.
    production_anchor = '    cache = _v28_load()\n    representatives = {}\n'
    production_replacement = r'''    cache = _v28_load()
    _one_key = _v24c_shape_key(frozenset((V23C_ROOT,)))
    if _one_key in V28_SHAPE_KEYS and _one_key not in cache:
        cache[_one_key] = V28_ONE_FACE
        _v28_save(cache)
        print("  seeded order-native one-face smoke into production checkpoint")
    representatives = {}
'''
    source = source.replace(production_anchor, production_replacement, 1)

    # Remove the physical O4 support-corpus regeneration. Generate the requested order directly.
    o4_start_marker = '    print("\\n[3] GENERIC SUPPORT-CENSUS REGRESSION AT ORDER FOUR")'
    order_start_marker = '    print(f"\\n[4] ORDER-{V28_ORDER} SUPPORT CENSUS")'
    o4_start = source.index(o4_start_marker)
    order_start = source.index(order_start_marker, o4_start)
    direct_order_block = r'''    print(f"\n[3] DIRECT ORDER-{V28_ORDER} HISTORY GENERATION — O4 RECHECK SKIPPED")
    if _v28_census_cache is None:
        _v28_histories = _v28_history_levels(V28_SUPPORT_HALF_DEPTH, V23C_POL)
        V28_O4_SUPPORTS = set()
        V28_O4_STATS = {"skipped": True, "reason": "M5-first runner"}
    else:
        _v28_histories = None
        V28_O4_SUPPORTS = set(_v28_census_cache.get("o4_supports", ()))
        V28_O4_STATS = dict(_v28_census_cache.get("o4_stats", {}))
        print(f"  loaded order-{V28_ORDER} census checkpoint; no O4 regeneration")

'''
    source = source[:o4_start] + direct_order_block + source[order_start:]

    # Make the banner and result provenance explicit.
    banner = 'print("watchdog                     : NONE")\n'
    source = source.replace(
        banner,
        banner + 'print("m4 physical/operator rerun   : SKIPPED")\nprint("support-census start order   :", V28_ORDER)\n',
        1,
    )

    required_absent = (
        '[2b] ONE-FACE PHYSICAL PREFIX REGRESSION',
        'GENERIC SUPPORT-CENSUS REGRESSION AT ORDER FOUR',
        'through O4--O7',
        'completed v10a.26 one-face comparator is unavailable',
        '_v23c_fit_cluster(',
        '"_v10a3_h0_state_inner"',
        '"_v26_sw_blocks"',
    )
    leftovers = [x for x in required_absent if x in source]
    if leftovers:
        raise RuntimeError(f"M5-first v28 patch incomplete; leftovers={leftovers}")
    if source == original:
        raise RuntimeError("v28 patch made no changes")
    return source


V28_ORIGINAL_SOURCE = V28_PATH.read_text(encoding="utf-8", errors="strict")
if 'target coefficient           : NOT LOADED' not in V28_ORIGINAL_SOURCE:
    raise RuntimeError("v10a.28 blind-target declaration is absent")

V28_PATCHED_SOURCE = patch_v28_m5_first(V28_ORIGINAL_SOURCE)
V28_PATCHED_PATH = SOURCE_DIR / f"Hodge_v10a30_M5_FIRST_engine_{V28_SHA256[:12]}.py"
V28_PATCHED_PATH.write_text(V28_PATCHED_SOURCE, encoding="utf-8")
V28_PATCHED_SHA256 = sha256_file(V28_PATCHED_PATH)

# Static compilation before touching production.
compile(V28_PATCHED_SOURCE, str(V28_PATCHED_PATH), "exec")
print("Patched engine:", V28_PATCHED_PATH)
print("Patched SHA-256:", V28_PATCHED_SHA256)
print("Separate m4 physical validation: REMOVED")
print("Order-4 support census: REMOVED")
print("Requested-order target values: ABSENT")



## Start here: blind \(m_5\) test and production

This is the first scientific run. It executes the patched engine once in order-5 production mode. The order-5 firewall, order-5 support census, and rooted production occur in the same invocation; there is no duplicate preflight invocation.


In [ ]:

def configure_order(order: int, *, max_new_shapes: int, time_budget_minutes: float):
    order = int(order)
    if order not in (5, 6, 7):
        raise ValueError("This M5-first runner supports orders 5, 6, and 7")
    cap = 7 if order == 5 else 9
    order_dir = WORKDIR / f"m{order}"
    order_dir.mkdir(parents=True, exist_ok=True)

    settings = {
        "V28_ORDER": str(order),
        "V28_MODE": "production",
        "V28_HAAR_CAP": str(cap),
        "V28_RUN_CENSUS": "1",
        "V28_GPU": "1",
        "V28_GPU_SW_MIN_DIM": str(GPU_SW_MIN_DIM),
        "V28_HERMITICITY_AUDIT_PAIRS": str(HERM_AUDIT_PAIRS),
        "V28_DUPLICATE_CHECKS": str(DUPLICATE_CHECKS),
        "V28_HEARTBEAT": str(HEARTBEAT_SECONDS),
        "V28_RESUME": "1",
        "V28_ALLOW_REFERENCE_REBUILD": "0",
        "V28_PRODUCTION_CONFIRM": f"YES_ORDER_{order}",
        "V28_MAX_NEW_SHAPES": str(int(max_new_shapes)),
        "V28_TIME_BUDGET_MINUTES": str(float(time_budget_minutes)),
        "V28_CHECKPOINT": str(order_dir / "m5_first_shapes.pkl"),
        "V28_CENSUS_CHECKPOINT": str(order_dir / "m5_first_census.pkl"),
    }
    os.environ.update(settings)
    return order_dir, settings


def execute_order(order: int, *, max_new_shapes: int, time_budget_minutes: float):
    order_dir, settings = configure_order(
        order, max_new_shapes=max_new_shapes, time_budget_minutes=time_budget_minutes,
    )
    # Remove prior execution products, but retain caches internal to definitions.
    for name in tuple(globals()):
        if name.startswith("V28_") and name not in {
            "V28_PATH", "V28_SHA256",
            "V28_ORIGINAL_SOURCE", "V28_PATCHED_SOURCE",
            "V28_PATCHED_PATH", "V28_PATCHED_SHA256",
        }:
            globals().pop(name, None)
    print("=" * 100)
    print(f"BEGIN BLIND M{order} — NO M4 PHYSICAL RERUN")
    print("=" * 100)
    t0 = time.time()
    exec(compile(V28_PATCHED_SOURCE, str(V28_PATCHED_PATH), "exec"), globals())
    elapsed = time.time() - t0
    result = globals().get("V28_RESULT")
    if result is None:
        raise RuntimeError("Patched engine did not return V28_RESULT")
    print(f"Order-{order} invocation elapsed: {elapsed / 3600:.3f} h")
    print("Complete:", result.get("complete", True))
    return result, elapsed, order_dir, settings


V28_RESULT, elapsed_m5, M5_DIR, M5_SETTINGS = execute_order(
    5,
    max_new_shapes=M5_MAX_NEW_SHAPES,
    time_budget_minutes=M5_TIME_BUDGET_MINUTES,
)

if not V28_RESULT.get("complete", True):
    print("Atomic checkpoint saved. Rerun this cell to continue from the completed shapes.")


In [ ]:

# Freeze blind m5 when complete.
def _jsonable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    return x

M5_SUMMARY = M5_DIR / "blind_m5_summary.json"
if V28_RESULT.get("complete", True):
    coeff = np.asarray(V28_RESULT["coefficients"], dtype=float)
    payload = {
        "schema": "hodge-v10a30-m5-first-blind-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": 5,
        "blind": True,
        "external_requested_order_target_loaded": False,
        "m4_physical_rerun": False,
        "o4_support_census_rerun": False,
        "coefficient_vector_m0_to_m5": coeff.tolist(),
        "m5": float(coeff[5]),
        "v10a26_source_sha256": V26_SHA256,
        "v10a26_bootstrap_sha256": sha256_file(BOOTSTRAP_PATH),
        "v10a28_source_sha256": V28_SHA256,
        "m5_first_engine_sha256": V28_PATCHED_SHA256,
        "v28_schema": V28_RESULT.get("schema"),
        "v28_signature": V28_RESULT.get("signature"),
        "concrete_clusters": int(V28_RESULT.get("concrete_clusters", 0)),
        "shape_classes": int(V28_RESULT.get("shapes", 0)),
        "haar_cap": int(V28_RESULT.get("haar_cap", 0)),
        "krylov_depth": int(V28_RESULT.get("krylov_depth", 0)),
        "gates": [
            {"name": n, "passed": bool(ok), "detail": d}
            for n, ok, d in globals().get("V28_GATES", [])
        ],
        "gpu": GPU_NAME,
        "elapsed_seconds_last_invocation": float(elapsed_m5),
        "checkpoint": M5_SETTINGS["V28_CHECKPOINT"],
        "census_checkpoint": M5_SETTINGS["V28_CENSUS_CHECKPOINT"],
    }
    M5_SUMMARY.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("FROZEN BLIND m5:", repr(float(coeff[5])))
    print("Summary:", M5_SUMMARY)
    print("Summary SHA-256:", sha256_file(M5_SUMMARY))
else:
    print("m5 is incomplete; summary will be frozen only after all shapes finish.")



## External holdout boundary

Compare the frozen JSON to the published \(m_5\) value outside this runtime. The target is not present in this notebook.


In [ ]:

MARK_M5_EXTERNALLY_VALIDATED = False

flag5 = M5_DIR / "M5_EXTERNALLY_VALIDATED.flag"
if MARK_M5_EXTERNALLY_VALIDATED:
    if not M5_SUMMARY.exists():
        raise RuntimeError("blind_m5_summary.json is missing")
    flag5.write_text(
        "Frozen blind m5 marked externally validated at "
        + datetime.now(timezone.utc).isoformat() + "\n",
        encoding="utf-8",
    )
    print("Created:", flag5)
else:
    print("m6 remains locked until the frozen m5 is externally checked.")



## Resumable \(m_6\) and \(m_7\)

These use the same definitions-only bootstrap and current-order-only engine. They do not rerun the physical fourth-order notebook.


In [ ]:

def freeze_higher_order(order: int, result: dict, elapsed: float, order_dir: Path, settings: dict):
    coeff = np.asarray(result["coefficients"], dtype=float)
    summary = order_dir / f"blind_m{order}_summary.json"
    lower = {}
    for k in range(5, order):
        prior = WORKDIR / f"m{k}" / f"blind_m{k}_summary.json"
        if prior.exists():
            old = json.loads(prior.read_text(encoding="utf-8"))
            lower[f"m{k}"] = {
                "current": float(coeff[k]),
                "frozen": float(old[f"m{k}"]),
                "abs_error": abs(float(coeff[k]) - float(old[f"m{k}"])),
            }
    payload = {
        "schema": f"hodge-v10a30-m5-first-blind-m{order}-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": order,
        "blind": True,
        "external_requested_order_target_loaded": False,
        "m4_physical_rerun": False,
        f"m{order}": float(coeff[order]),
        "coefficient_vector": coeff.tolist(),
        "lower_order_self_consistency": lower,
        "v10a26_source_sha256": V26_SHA256,
        "v10a26_bootstrap_sha256": sha256_file(BOOTSTRAP_PATH),
        "v10a28_source_sha256": V28_SHA256,
        "m5_first_engine_sha256": V28_PATCHED_SHA256,
        "v28_schema": result.get("schema"),
        "v28_signature": result.get("signature"),
        "gates": [
            {"name": n, "passed": bool(ok), "detail": d}
            for n, ok, d in globals().get("V28_GATES", [])
        ],
        "gpu": GPU_NAME,
        "elapsed_seconds_last_invocation": float(elapsed),
        "checkpoint": settings["V28_CHECKPOINT"],
        "census_checkpoint": settings["V28_CENSUS_CHECKPOINT"],
    }
    summary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"FROZEN BLIND m{order}:", repr(float(coeff[order])))
    print("Summary:", summary)
    print("Summary SHA-256:", sha256_file(summary))
    return summary


def run_higher_order(order: int, *, max_new_shapes: int = 1, time_budget_minutes: float = 30):
    order = int(order)
    if order not in (6, 7):
        raise ValueError("order must be 6 or 7")
    if not (WORKDIR / "m5" / "M5_EXTERNALLY_VALIDATED.flag").exists():
        raise RuntimeError("m6/m7 locked: m5 is not externally validated")
    if order == 7 and not (WORKDIR / "m6" / "M6_EXTERNALLY_VALIDATED.flag").exists():
        raise RuntimeError("m7 locked: m6 is not externally validated")

    result, elapsed, order_dir, settings = execute_order(
        order,
        max_new_shapes=max_new_shapes,
        time_budget_minutes=time_budget_minutes,
    )
    if not result.get("complete", True):
        print(f"m{order} checkpoint saved; rerun this helper to continue")
        return result
    freeze_higher_order(order, result, elapsed, order_dir, settings)
    return result

# After validation flags exist:
# run_higher_order(6)
# run_higher_order(7)


In [ ]:

# Compact provenance bundle; large .pkl checkpoints remain in Drive and are excluded.
bundle = WORKDIR / "blind_results_m5_first_compact.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORKDIR.rglob("*")):
        if not p.is_file() or p.suffix == ".pkl" or p == bundle:
            continue
        zf.write(p, p.relative_to(WORKDIR))
    zf.write(V26_PATH, Path("original_sources") / V26_PATH.name)
    zf.write(V28_PATH, Path("original_sources") / V28_PATH.name)
print("Compact bundle:", bundle)
print("SHA-256:", sha256_file(bundle))



## Return after the run

Send `blind_m5_summary.json`, the final gate summary, and the output around `ROOTED INCIDENCE TRANSFORM`. The old full-\(T_1\) fourth-order calculation should never appear in this notebook's output.
